In [ ]:
import json
with open("app data/data.json", "r", encoding="utf-8") as f:
    form_data_dict = json.load(f)

with open("app data/prompts.json", "r", encoding="utf-8") as f:
    prompts_dict = json.load(f)



In [ ]:
from typing import Any, Dict



def attach_prompts_to_output(
    output_dict: Dict[str, Any],
    prompts_dict: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Attach structured prompts (4 fixed keys) to the output.

    - Leaves → {"value": ..., "prompts": {...}}
    - Non-leaf nodes may also contain "prompts"
    - Root structure is preserved
    """

    def is_prompt_object(node: Any) -> bool:
        """Check if node contains the 4 structured prompt fields"""
        return (
            isinstance(node, dict)
            and {
                "admissibility_guide",
                "application_notes",
                "common_mistakes",
                "case_law",
            }.issubset(node.keys())  # more flexible than exact match
        )

    def recurse(output_node: Any, prompt_node: Any) -> Any:
        # Case 1: Dictionary → recurse deeper
        if isinstance(output_node, dict):
            merged: Dict[str, Any] = {}

            for key, value in output_node.items():
                child_prompt = (
                    prompt_node.get(key)
                    if isinstance(prompt_node, dict)
                    else None
                )
                merged[key] = recurse(value, child_prompt)

            # Attach prompts at this level (if applicable)
            if is_prompt_object(prompt_node):
                merged["prompts"] = prompt_node

            return merged

        # Case 2: Leaf node
        return {
            "value": output_node,
            "prompts": prompt_node if is_prompt_object(prompt_node) else None,
        }

    return recurse(output_dict, prompts_dict)



In [ ]:
attach_prompts_to_output(form_data_dict, prompts_dict)

{'A. Applicant': {'A.1 Individual': {'Surname': {'value': 'D',
    'prompts': {'admissibility_guide': 'The applicant must be identifiable as a specific individual. Applications attributed to no identified person are treated as anonymous and declared inadmissible under Article 35 § 2(a). Rule 47 requires full identification of every applicant.',
     'application_notes': 'Write your family name exactly as it appears in your official identity document (passport or national ID card). Use the standard form used in official documents in your country.',
     'common_mistakes': 'Mistake no. 1: Not using the Court’s current application form. If the form is outdated, the name fields may not link correctly to the Court’s database system. Always download the latest form from www.echr.coe.int.',
     'case_law': 'No specific ECtHR judgment addresses this administrative field directly. The general principle from Malysh and Ivanin v. Ukraine applies: any incompleteness in the application form means 

In [ ]:
# form_data_with_prompts_dict= attach_prompts_to_output(form_data_dict, prompts_dict)

In [ ]:
import json
from pathlib import Path

form_data_with_prompts_dict= attach_prompts_to_output(form_data_dict, prompts_dict)

folder = Path("app data")
folder.mkdir(exist_ok=True)

file_path = folder / "form_data_with_prompts.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(form_data_with_prompts_dict, f, indent=2)


: 